# Cryptic IP Binding Sites — **Complete Pipeline** (Google Colab)

One notebook to run the **entire** post-Phase-1 workflow end-to-end:

| Step | What it does |
|------|----------------|
| **Install** | fpocket, FreeSASA, APBS, Python deps |
| **Tier-1 gate** | ADAR2 vs PLCδ1 validation (must pass) |
| **ML benchmark** | Pocket-level classifier train + ROC comparison |
| **Yeast pilot** | AlphaFold proteome screen (25 → 500 proteins) |
| **Publication** | Controls, figures (PDF/PNG), methods, provenance |
| **MD pilot** | Optional short OpenMM runs (full preset only) |
| **Export** | Zip all results for download |

**Runtime:** CPU is fine (GPU not required). Use **High-RAM** for `pilot` preset.

**Presets:**
- `quick` — 25 proteins, ~15–30 min (smoke test)
- `pilot` — 500 proteins, ~2–4 h (recommended)
- `full` — 500 proteins + MD pilot, ~3–5 h

Open in Colab: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/cryptic-ip-binding-sites/blob/cursor/full-pipeline-improvements-6dd9/notebooks/Colab_Full_Pipeline_Run.ipynb)

## 0. Optional — mount Google Drive (recommended for `pilot` / `full`)

Uncomment the next cell to save results to Drive instead of ephemeral Colab disk.

In [ ]:
USE_DRIVE = False  # set True for long runs
DRIVE_SUBDIR = 'cryptic_ip_results'

from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_BASE = Path('/content/drive/MyDrive') / DRIVE_SUBDIR
else:
    OUTPUT_BASE = Path('results/colab_run')

print('Results will be written to:', OUTPUT_BASE)

## 1. Clone repository and install dependencies

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = 'cryptic-ip-binding-sites'
BRANCH = 'cursor/full-pipeline-improvements-6dd9'  # switch to 'main' after merge

if IN_COLAB:
    if Path(REPO).exists():
        !cd {REPO} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
    else:
        !git clone -b {BRANCH} https://github.com/Tommaso-R-Marena/cryptic-ip-binding-sites.git {REPO}
    os.chdir(REPO)
    !bash scripts/colab_install.sh
else:
    root = Path('.').resolve()
    if not (root / 'cryptic_ip').exists() and (root.parent / 'cryptic_ip').exists():
        os.chdir(root.parent)

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
print('Working directory:', ROOT)
print('Branch:', BRANCH)

## 2. Configuration

Change `PRESET` only — everything else is handled by `scripts/colab_run_all.py`.

In [ ]:
from pathlib import Path

# --- Choose one preset ---
PRESET = 'quick'   # 'quick' | 'pilot' | 'full'

# Override preset defaults (optional)
N_PROTEINS = None       # e.g. 50; None = use preset default
WORKERS = 2             # Colab free tier: 2 CPUs
SCORE_THRESHOLD = 0.75  # cryptic candidate gate
WITH_ELECTROSTATICS = False  # True = APBS per structure (3-5x slower)
SKIP_DOWNLOAD = False     # True = reuse cached AlphaFold structures
RUN_MD = None             # None = follow preset; True/False to override

if 'OUTPUT_BASE' not in globals():
    OUTPUT_BASE = Path('results/colab_run')
OUTPUT_DIR = Path(OUTPUT_BASE)
STRUCTURES_DIR = ROOT / 'data' / 'structures' / 'yeast_pilot'

print('Preset:', PRESET)
print('Output:', OUTPUT_DIR.resolve())

## 3. Run the complete pipeline

This single command runs **all stages** with logging and a final zip archive.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, 'scripts/colab_run_all.py',
    '--preset', PRESET,
    '--output-dir', str(OUTPUT_DIR),
    '--structures-dir', str(STRUCTURES_DIR),
    '--score-threshold', str(SCORE_THRESHOLD),
    '--workers', str(WORKERS),
]
if N_PROTEINS is not None:
    cmd.extend(['--n-proteins', str(N_PROTEINS)])
if WITH_ELECTROSTATICS:
    cmd.append('--with-electrostatics')
if SKIP_DOWNLOAD:
    cmd.append('--skip-download')
if RUN_MD is False:
    cmd.append('--skip-md')

print('Running:', ' '.join(cmd))
proc = subprocess.run(cmd, cwd=ROOT)
if proc.returncode != 0:
    raise RuntimeError(f'Pipeline failed with exit code {proc.returncode}. See {OUTPUT_DIR}/colab_run.log')

## 4. Results summary

In [ ]:
import json
from IPython.display import Markdown, display
import pandas as pd

manifest_path = OUTPUT_DIR / 'run_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print(json.dumps(manifest, indent=2))

summary_md = OUTPUT_DIR / 'publication' / 'RESULTS_SUMMARY.md'
if summary_md.exists():
    display(Markdown(summary_md.read_text()))

yeast_summary = OUTPUT_DIR / 'yeast_pilot' / 'yeast_pilot_summary.json'
if yeast_summary.exists():
    print('\nYeast pilot:')
    print(json.dumps(json.loads(yeast_summary.read_text()), indent=2))

ml_csv = OUTPUT_DIR / 'ml_training' / 'ml_vs_threshold_comparison.csv'
if ml_csv.exists():
    print('\nML vs threshold:')
    display(pd.read_csv(ml_csv))

hits_csv = OUTPUT_DIR / 'yeast_pilot' / 'yeast_pilot_hits.csv'
if hits_csv.exists():
    hits = pd.read_csv(hits_csv)
    print(f'\nTop yeast hits ({len(hits)} rows):')
    display(hits.sort_values('composite_score', ascending=False).head(15))

## 5. Publication figures

In [ ]:
from IPython.display import Image, display

fig_dir = OUTPUT_DIR / 'publication' / 'figures'
for name in ['Figure1_Overview', 'Figure2_Comparative_Proteomics', 'Figure3_Top_Candidates']:
    png = fig_dir / f'{name}.png'
    if png.exists():
        print(name)
        display(Image(filename=str(png), width=700))
    else:
        print(f'(missing {png.name})')

## 6. Download results (Colab)

If you mounted Drive, results are already persisted. Otherwise download the zip.

In [ ]:
if IN_COLAB and not USE_DRIVE:
    archive = OUTPUT_DIR.parent / 'colab_pipeline_results.zip'
    if not archive.exists():
        !cd {OUTPUT_DIR.parent} && zip -r colab_pipeline_results.zip {OUTPUT_DIR.name}
    from google.colab import files
    files.download(str(archive))
elif IN_COLAB and USE_DRIVE:
    print('Results saved to Drive:', OUTPUT_DIR)
else:
    print('Local run complete:', OUTPUT_DIR)

## 7. Advanced — run individual stages

Use these if you need to re-run a single step without repeating the full pipeline.

In [ ]:
# Example: re-run yeast pilot only (after fixing parameters)
RUN_STAGE = None  # set to 'yeast', 'ml', 'publication', or 'tier1'

if RUN_STAGE == 'tier1':
    !python scripts/colab_run_all.py --preset quick --skip-ml --skip-yeast --skip-publication --skip-md --skip-package --output-dir {OUTPUT_DIR}
elif RUN_STAGE == 'ml':
    !python scripts/train_ml_classifier.py --skip-build-dataset --work-dir {OUTPUT_DIR}/ml_training --model-dir models
elif RUN_STAGE == 'yeast':
    !python scripts/run_yeast_pilot_screen.py --n-proteins {N_PROTEINS or 25} --workers {WORKERS} --score-threshold {SCORE_THRESHOLD} --output-dir {OUTPUT_DIR}/yeast_pilot --structures-dir {STRUCTURES_DIR} --skip-download
elif RUN_STAGE == 'publication':
    !python scripts/run_publication_package.py --output-dir {OUTPUT_DIR}/publication --skip-dataset-build
else:
    print('Set RUN_STAGE to re-run a single stage.')